In [1]:
from google.colab import files
import pandas as pd

# 파일 업로드
uploaded = files.upload()

# 데이터프레임으로 파일 읽기
df = pd.read_csv('/content/cleaned_labeling_[SEP]Delete_sample.csv')

Saving cleaned_labeling_[SEP]Delete_sample.csv to cleaned_labeling_[SEP]Delete_sample.csv


In [2]:
import re

def parse_dialogue(dialogue_text):
    # speaker 구분이 '0:' 또는 '1:' 으로 되어있으므로 split
    # 단, 첫번째 '0:' 또는 '1:' 앞에 불필요한 공백 가능성 있음 → re 사용

    # '0:' 또는 '1:' 앞에서 split (keep separator)
    splits = re.split(r'(?=(?:0:|1:))', dialogue_text)

    # 빈 split 제거 + strip 적용
    turns = [s.strip() for s in splits if s.strip() != '']

    return turns

In [3]:
df['text'] = df['text'].apply(parse_dialogue)

In [4]:
# 학습용 데이터 구성

# window_size : 모델에 넣을 대화 흐름 길이 결정
def create_flow_inputs(turns_list, labels, window_size=10):
    input_texts = []
    target_labels = []

    for turns, label in zip(turns_list, labels):
        # 대화 turn 수가 window_size 미만이면 skip
        if len(turns) < window_size:
            continue

        # sliding window 적용 -> window_size 크기로 대화 잘라서 여러 개 샘플 생성
        # window가 한 칸씩 앞으로 밀리면서 데이터 생성(0 ~ 10, 1 ~ 11)
        for i in range(window_size, len(turns) + 1):
            window_turns = turns[i - window_size:i]
            model_input_text = ' [SEP] '.join(window_turns)

            input_texts.append(model_input_text)
            target_labels.append(label)

    return input_texts, target_labels

# 구성
input_texts, target_labels = create_flow_inputs(df['text'], df['label'], window_size=10)

# Huggingface Dataset 구성
from datasets import Dataset
dataset = Dataset.from_dict({'text': input_texts, 'label': target_labels})

# Tokenizer
from transformers import BartTokenizerFast
tokenizer = BartTokenizerFast.from_pretrained('facebook/bart-base')

def tokenize_function(examples):
    return tokenizer(examples['text'], max_length=512, truncation=True, padding='max_length')

dataset = dataset.map(tokenize_function, batched=True)

# Train / Validation Split
train_test_split = dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
val_dataset = train_test_split['test']


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

Map:   0%|          | 0/779 [00:00<?, ? examples/s]

In [5]:
import torch
import torch.nn as nn
from transformers import BartModel

# PyTorch의 nn.Module 상속받아 DialogBARTForGroomingDetection 모델생성
class DialogBARTForGroomingDetection(nn.Module):
    # model_name='facebook/bart-base' : Huggingface Hub에서 사전학습된 facebook/bart-base
    def __init__(self, model_name='facebook/bart-base', num_labels=2):
        # 부모 클래스 (nn.Module)의 __init__() 호출해서 nn.Module 내부 초기화 수행
        super(DialogBARTForGroomingDetection, self).__init__()
        # BARTModel 로드
        # BART의 Encoder output만 사용 (Decoder 사용 안 함 → Language Generation X → Classification O)
        self.bart = BartModel.from_pretrained(model_name)

        # hidden_size : BART Encoder output의 마지막 hidden state 크기 (768 for base)
        # num_labels : 이진분류(Grooming / Non-Grooming)
        self.classifier = nn.Linear(self.bart.config.hidden_size, num_labels)

    # input_ids: Tokenized input ids (Tensor)
    # attention_mask: Attention mask (padding 무시 처리용)
    # labels: (optional) -정답 label (0/1) → 학습 시 loss 계산용
    def forward(self, input_ids, attention_mask, labels=None):
        # BARTModel 에 input_ids / attention_mask 전달 → forward pass 수행
        outputs = self.bart(input_ids=input_ids, attention_mask=attention_mask)

        # BART는 기본적으로 [CLS] token이 없지만 → 첫 번째 token 위치의 hidden state 사용
        hidden_state = outputs.last_hidden_state[:, 0, :]  # First token 사용

        # hidden_state → Linear Layer 통과 → logits (batch_size, num_labels)
        logits = self.classifier(hidden_state)

        # labels 가 주어졌을 때만 Loss 계산 → 학습/검증 시에만 loss 사용
        # Loss Function → CrossEntropyLoss → Softmax + NLL Loss 포함
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        # 'loss': 학습 시 사용
        # 'logits': 예측 시 사용 (Softmax 적용 후 확률값 계산 가능)
        return {'loss': loss, 'logits': logits}


In [6]:
# 학습 실행 (Trainer 사용)

from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_safetensors=False  # BART는 False 권장
)

model = DialogBARTForGroomingDetection()  # 커스텀 모델 사용

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 학습 시작
trainer.train()

# 모델 저장
trainer.save_model("./dialogbart_grooming_model")
tokenizer.save_pretrained("./dialogbart_grooming_model")


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

<ipython-input-6-67f013d27e7e>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shjeong020208 (shjeong020208-dong-eui-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.424500,0.268385
2,0.158500,0.084554
3,0.045500,0.048622


('./dialogbart_grooming_model/tokenizer_config.json',
 './dialogbart_grooming_model/special_tokens_map.json',
 './dialogbart_grooming_model/vocab.json',
 './dialogbart_grooming_model/merges.txt',
 './dialogbart_grooming_model/added_tokens.json',
 './dialogbart_grooming_model/tokenizer.json')

In [7]:
# Threshold tuning

from sklearn.metrics import precision_recall_fscore_support
import numpy as np

# validation prediction
outputs = trainer.predict(val_dataset)
logits = outputs.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
probs_grooming = probs[:, 1]

true_labels = val_dataset['label']

# Threshold search
thresholds = np.arange(0.1, 0.95, 0.05)
best_f1 = 0
best_threshold = 0

for threshold in thresholds:
    pred_labels = (probs_grooming >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')
    print(f"Threshold {threshold:.2f} → Precision {precision:.3f}, Recall {recall:.3f}, F1 {f1:.3f}")

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"\n✅ Best threshold found: {best_threshold:.2f} with F1-score: {best_f1:.3f}")


Threshold 0.10 → Precision 0.903, Recall 1.000, F1 0.949
Threshold 0.15 → Precision 0.927, Recall 1.000, F1 0.962
Threshold 0.20 → Precision 0.944, Recall 1.000, F1 0.971
Threshold 0.25 → Precision 0.953, Recall 1.000, F1 0.976
Threshold 0.30 → Precision 0.962, Recall 1.000, F1 0.981
Threshold 0.35 → Precision 0.962, Recall 1.000, F1 0.981
Threshold 0.40 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.45 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.50 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.55 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.60 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.65 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.70 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.75 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.80 → Precision 0.981, Recall 1.000, F1 0.990
Threshold 0.85 → Precision 0.990, Recall 1.000, F1 0.995
Threshold 0.90 → Precision 0.990, Recall 1.000, F1 0.995

✅ Best threshold found: 0.85 w

In [8]:
# 모델 로드
model = DialogBARTForGroomingDetection()
model.load_state_dict(torch.load('/content/dialogbart_grooming_model/pytorch_model.bin'))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

tokenizer = BartTokenizerFast.from_pretrained("/content/dialogbart_grooming_model")

# 실시간 대화 흐름 관리
dialogue_history = []
window_size = 10
best_threshold = 0.7  # tuning 결과 적용

import time
last_update_time = time.time()
reset_timeout = 180  # 3분 inactivity 시 history reset

print("=== Grooming 위험도 실시간 탐지 시작 ===")
print("※ 상대방 발화는 '1:' 으로 시작, 내 발화는 '0:' 으로 시작해서 입력해주세요.")
print("※ 'END' 입력 시 종료\n")

while True:
    new_utterance = input("새로운 채팅 입력 (형식: '0: hello' 또는 '1: hi'): ")
    if new_utterance == "END":
        break

    # Inactivity check
    current_time = time.time()
    if current_time - last_update_time > reset_timeout:
        print("💡 대화 inactivity → dialogue_history reset됨.")
        dialogue_history = []

    last_update_time = current_time

    # 입력 validation
    if not (new_utterance.startswith("0:") or new_utterance.startswith("1:")):
        print("⚠️ 입력은 반드시 '0:' 또는 '1:' 으로 시작해야 합니다.\n")
        continue

    dialogue_history.append(new_utterance.strip())
    if len(dialogue_history) > window_size:
        dialogue_history = dialogue_history[-window_size:]

    model_input_text = ' [SEP] '.join(dialogue_history)
    inputs = tokenizer(model_input_text, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])
        logits = outputs['logits']
        probs = torch.softmax(logits, dim=-1)

    grooming_prob = probs[:, 1].item()
    print(f"\n⚠️ Grooming 위험도 (대화 흐름 기준): {grooming_prob:.3f}")

    if grooming_prob >= best_threshold:
        print("🚨 경고: Grooming 위험 가능성 있음!\n")
    else:
        print("안전합니다.\n")


=== Grooming 위험도 실시간 탐지 시작 ===
※ 상대방 발화는 '1:' 으로 시작, 내 발화는 '0:' 으로 시작해서 입력해주세요.
※ 'END' 입력 시 종료

새로운 채팅 입력 (형식: '0: hello' 또는 '1: hi'): 1: hi

⚠️ Grooming 위험도 (대화 흐름 기준): 0.090
안전합니다.

새로운 채팅 입력 (형식: '0: hello' 또는 '1: hi'): 0: hi

⚠️ Grooming 위험도 (대화 흐름 기준): 0.621
안전합니다.

새로운 채팅 입력 (형식: '0: hello' 또는 '1: hi'): END
